# 스태킹(Stacking) — 튜닝한 모델 3개를 하나로

수업 필기 `분류 모델 성능 고도화 실습 1` 의 마지막 부분(Stacking 분류 모델)을 정리 · 개선한 노트입니다.
(교재 `머신러닝` v7 p.109~114 — 스태킹 · 모델 선택 기준)

- 기본 모델(base) 3개는 [`hyperparameter_search_rf_xgb.ipynb`](./hyperparameter_search_rf_xgb.ipynb) 에서 튜닝한 **결정 트리 · 랜덤 포레스트 · XGBoost**.
- 원본에서 고친 부분은 **📝 고친 부분** 으로 따로 적어 두었습니다.

In [1]:
# 그래프 한글 폰트 설정 (운영체제별)
import platform
import matplotlib.pyplot as plt

if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'Noto Sans CJK JP'
plt.rcParams['axes.unicode_minus'] = False

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# 데이터 불러오기 (노트북과 같은 폴더)
df = pd.read_csv('./diabetes_binary_5050split_health_indicators_BRFSS2015.csv')

X_data = df.drop(columns=['Diabetes_binary'])   # 특성 21개
y_data = df['Diabetes_binary']                   # 정답 0.0 / 1.0

# 학습용 80% : 평가용 20% — 수업과 같은 분할
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.2, random_state=0, stratify=y_data)
print("학습용 :", X_train.shape, " 평가용 :", X_test.shape)

학습용 : (56553, 21)  평가용 : (14139, 21)


In [3]:
# 튜닝한 모델 3개 — hyperparameter_search_rf_xgb.ipynb 에서 저장한 파일이 있으면 불러오고,
# 없으면 거기서 찾은 최적 조합으로 바로 학습한다 (탐색을 다시 돌리지 않음)
import os, joblib
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

def load_or_fit(name, model):
    path = f'./models/{name}.pkl'
    if os.path.exists(path):
        return joblib.load(path)
    return model.fit(X_train, y_train)

best_dt  = load_or_fit('best_dt',  DecisionTreeClassifier(max_depth=7, min_samples_leaf=5, random_state=0))
best_rf  = load_or_fit('best_rf',  RandomForestClassifier(**{'n_estimators': 400, 'max_depth': 9, 'min_samples_leaf': 2}, random_state=0))
best_xgb = load_or_fit('best_xgb', XGBClassifier(**{'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 4}, random_state=0))
for m in [best_dt, best_rf, best_xgb]:
    print(type(m).__name__, {k: m.get_params()[k] for k in ['max_depth', 'min_samples_leaf', 'n_estimators', 'learning_rate'] if k in m.get_params()})

DecisionTreeClassifier {'max_depth': 7, 'min_samples_leaf': 5}
RandomForestClassifier {'max_depth': 9, 'min_samples_leaf': 2, 'n_estimators': 400}
XGBClassifier {'max_depth': 4, 'n_estimators': 200, 'learning_rate': 0.05}


## 1. 스태킹의 구조 (교재 p.109~110)

| 단계 | 학습할 때 | 예측할 때 |
|---|---|---|
| 1단계 기본 모델 (Base) | 서로 다른 알고리즘으로 각각 학습 | 각자 예측 → **예측값 3개** |
| 2단계 메타 모델 (Meta) | **기본 모델들의 예측값을 X**, 원래 정답을 y 로 학습 | 예측값 3개를 받아 **최종 예측** |

보팅(다수결 · 평균)은 "모두 똑같이 한 표" 지만, 스태킹은 **누구 말을 얼마나 믿을지를 메타 모델이 학습**한다.

In [4]:
import time
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import StackingClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

# 1단계 : 앞에서 고도화를 완료한 개별 모델
base_models = [('dt', best_dt), ('rf', best_rf), ('xgb', best_xgb)]

# 2단계 : 기본 모델들의 예측값을 받아 최종 판단을 내리는 메타 모델
final_meta = LogisticRegression(max_iter=1000)

stack = StackingClassifier(estimators=base_models, final_estimator=final_meta,
                           cv=5,          # 기본값 — 아래 2절에서 설명
                           n_jobs=-1)
t0 = time.time()
stack.fit(X_train, y_train)
print(f"학습 완료 — {time.time()-t0:.0f}초")

pred_test_stack = stack.predict(X_test)
print(classification_report(y_test, pred_test_stack, digits=4))

학습 완료 — 41초


              precision    recall  f1-score   support

         0.0     0.7676    0.7308    0.7488      7070
         1.0     0.7431    0.7788    0.7605      7069

    accuracy                         0.7548     14139
   macro avg     0.7554    0.7548    0.7547     14139
weighted avg     0.7554    0.7548    0.7547     14139



In [5]:
# 기본 모델 각각 vs 스태킹 (평가용)
rows = []
for name, m in base_models + [('stack', stack)]:
    p = m.predict(X_test)
    rows.append({'모델': name, '평가 정확도': accuracy_score(y_test, p), '평가 F1': f1_score(y_test, p)})
pd.DataFrame(rows).round(4)

,모델,평가 정확도,평가 F1
0,dt,0.7411,0.7447
1,rf,0.7519,0.7597
2,xgb,0.7544,0.7629
3,stack,0.7548,0.7605


## 2. `fit` 안에서 실제로 일어나는 일 — **out-of-fold 예측**

교재 p.109 "학습용 데이터에 대한 개별 모델들의 예측 결과" 를 그대로 하면 문제가 생긴다. 기본 모델이 **이미 본 데이터**에 대해 예측하면 실제보다 훨씬 잘 맞혀서, 메타 모델이 "이 모델 말이 다 맞더라" 하고 **과하게 믿게** 된다.

그래서 `StackingClassifier` 는 어제 배운 **K-fold** 를 쓴다 :
1. 학습용을 5조각으로 나눠 → 4조각으로 학습한 기본 모델이 **나머지 1조각을 예측** (5번 반복) → 모든 행에 "**안 보고 낸 예측**"이 하나씩 생긴다 (`cross_val_predict`)
2. 이 out-of-fold 예측 3열로 메타 모델 학습
3. 기본 모델들은 **학습용 전체로 다시 학습** (평가용을 예측할 때 쓰는 모델)

예측값은 0/1 이 아니라 **확률**(`predict_proba` 의 1일 확률)이 들어간다 (`stack_method='auto'`).

In [6]:
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.base import clone

skf = StratifiedKFold(n_splits=5)      # StackingClassifier 의 cv=5 와 같은 분할
oof = pd.DataFrame(index=X_train.index)
for name, m in base_models:
    oof[name] = cross_val_predict(clone(m), X_train, y_train, cv=skf, method='predict_proba', n_jobs=-1)[:, 1]
print("메타 모델의 학습 데이터 (out-of-fold 확률 3열) :")
display(oof.head().round(3))

meta = LogisticRegression(max_iter=1000).fit(oof, y_train)
print("직접 만든 메타 모델 계수 :", meta.coef_.round(3), " 절편", meta.intercept_.round(3))
print("stack.final_estimator_  :", stack.final_estimator_.coef_.round(3), " 절편", stack.final_estimator_.intercept_.round(3))

# 평가용 : 학습용 전체로 다시 학습한 기본 모델들(stack.estimators_)의 확률 → 메타 모델
test_feat = pd.DataFrame({n: m.predict_proba(X_test)[:, 1] for (n, _), m in zip(base_models, stack.estimators_)})
my_pred = meta.predict(test_feat)
print("\n직접 만든 스태킹 예측 == StackingClassifier 예측 ? :", (my_pred == pred_test_stack).mean().round(4))

메타 모델의 학습 데이터 (out-of-fold 확률 3열) :


,dt,rf,xgb
35369,0.741,0.830,0.878
40967,0.492,0.359,0.313
27238,0.445,0.452,0.464
50582,0.749,0.628,0.740
38854,0.445,0.473,0.508


직접 만든 메타 모델 계수 : [[-0.149  1.609  3.897]]  절편 [-2.736]
stack.final_estimator_  : [[-0.149  1.609  3.897]]  절편 [-2.736]



직접 만든 스태킹 예측 == StackingClassifier 예측 ? : 1.0


- 메타 모델(로지스틱 회귀)의 계수 = **누구 말을 얼마나 믿나**. XGBoost **3.90** > 랜덤 포레스트 **1.61** ≫ 결정 트리 **−0.15** — 가장 약한 결정 트리는 사실상 무시된다 (두 모델이 이미 하는 말에 보탤 게 없다).
- 직접 만든 것과 `StackingClassifier` 가 같은 계수 · 같은 예측을 낸다 → `fit` 한 줄 안에 **교차 검증 + 메타 학습 + 재학습**이 들어 있다.

In [7]:
from matplotlib.animation import FuncAnimation, PillowWriter
import os
os.makedirs('./images', exist_ok=True)

# 🎬 애니메이션 : out-of-fold 예측표가 채워지는 과정 (보기 쉽게 학습용 50행만)
n_show = 50
idx_show = np.concatenate([val for _, val in skf.split(X_train, y_train)])  # fold 순서대로 정렬
fold_of = np.empty(len(X_train), dtype=int)
for f, (_, val) in enumerate(skf.split(X_train, y_train)):
    fold_of[val] = f
pick = np.sort(np.random.default_rng(0).choice(len(X_train), n_show, replace=False))
pick = pick[np.argsort(fold_of[pick], kind='stable')]
vals = oof.values[pick]          # (50, 3)
folds = fold_of[pick]
coef = meta.coef_[0]

fig, axes = plt.subplots(1, 3, figsize=(11, 4.8), gridspec_kw={'width_ratios': [1.1, 1.3, 1.4]})
def draw(frame):
    for ax in axes: ax.clear()
    f = min(frame, 5)
    # 왼쪽 : 학습용 5조각 — 지금 검증(예측 받는) 조각
    ax = axes[0]
    for k in range(5):
        color = '#2ca02c' if k == f else ('#bbbbbb' if f < 5 else '#9ecae1')
        ax.add_patch(plt.Rectangle((0, 4 - k), 1, 0.9, color=color))
        ax.text(0.5, 4 - k + 0.45, f'fold {k+1}' + ('  ← 예측' if k == f else '  학습' if f < 5 else ''), ha='center', va='center', fontsize=9)
    ax.set_xlim(0, 1); ax.set_ylim(0, 5); ax.axis('off')
    ax.set_title('학습용 56,553명' if f < 5 else '기본 모델은 전체로 재학습', fontsize=10)
    # 가운데 : out-of-fold 확률표
    ax = axes[1]
    shown = np.where(folds[:, None] < f, vals, np.nan) if f < 5 else vals
    ax.imshow(shown, cmap='coolwarm', vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(3), ['dt', 'rf', 'xgb']); ax.set_yticks([])
    ax.set_title(f'out-of-fold 확률 (50행 표본) — {min(f,5)}/5 조각 완료', fontsize=10)
    # 오른쪽 : 메타 모델
    ax = axes[2]
    if f >= 5:
        ax.barh(['dt', 'rf', 'xgb'], coef, color=['#aaaaaa', '#1f77b4', '#d62728'])
        for i, c in enumerate(coef): ax.text(c, i, f' {c:.2f}', va='center')
        ax.set_title('메타 모델(로지스틱 회귀) 계수\n= 누구 말을 얼마나 믿나', fontsize=10)
        ax.axvline(0, color='k', lw=0.8); ax.set_xlim(min(0, coef.min()) - 0.5, coef.max() * 1.3)
    else:
        ax.axis('off'); ax.text(0.5, 0.5, '표가 다 채워지면\n메타 모델 학습', ha='center', va='center', fontsize=11, color='gray')
    fig.tight_layout()

ani = FuncAnimation(fig, draw, frames=[0, 0, 1, 1, 2, 2, 3, 3, 4, 4, 5, 5, 6, 6, 6, 6], interval=600)
ani.save('./images/anim_03_stacking_oof.gif', writer=PillowWriter(fps=2), dpi=72)
plt.close()
print("저장 : ./images/anim_03_stacking_oof.gif")

저장 : ./images/anim_03_stacking_oof.gif


![스태킹 out-of-fold 애니메이션](./images/anim_03_stacking_oof.gif)

🎬 초록 조각을 예측할 때 기본 모델은 **나머지 4조각으로만** 학습한다. 5번 돌면 모든 행에 "안 보고 낸" 확률 3개가 채워지고, 그 표로 메타 모델을 학습한다.

### 2-1. out-of-fold 를 안 하면? — 이미 본 데이터로 만든 예측

차이가 잘 보이게 **기본값 결정 트리**(끝까지 자라서 학습용을 100% 외움)를 기본 모델에 넣어 본다.

In [8]:
from sklearn.tree import DecisionTreeClassifier

deep_dt = DecisionTreeClassifier(random_state=0)            # 기본값 — 학습용 정확도 ≈ 0.99
pair = [('deep_dt', deep_dt), ('xgb', best_xgb)]

# (X) 잘못된 방법 : 학습용 전체로 학습 → 같은 학습용을 예측한 확률로 메타 학습
fitted = [clone(m).fit(X_train, y_train) for _, m in pair]
in_sample = np.column_stack([m.predict_proba(X_train)[:, 1] for m in fitted])
meta_bad = LogisticRegression(max_iter=1000).fit(in_sample, y_train)
test_in = np.column_stack([m.predict_proba(X_test)[:, 1] for m in fitted])
acc_bad = accuracy_score(y_test, meta_bad.predict(test_in))

# (O) StackingClassifier : out-of-fold 확률로 메타 학습
stack_ok = StackingClassifier(estimators=pair, final_estimator=LogisticRegression(max_iter=1000), cv=5, n_jobs=-1).fit(X_train, y_train)
acc_ok = accuracy_score(y_test, stack_ok.predict(X_test))

print("기본값 트리 혼자   평가 정확도 :", round(accuracy_score(y_test, fitted[0].predict(X_test)), 4))
print("XGBoost 혼자       평가 정확도 :", round(accuracy_score(y_test, fitted[1].predict(X_test)), 4))
print(f"(X) 이미 본 데이터로 메타 학습 : 계수 {meta_bad.coef_.round(2)} → 평가 정확도 {acc_bad:.4f}")
print(f"(O) out-of-fold 로 메타 학습   : 계수 {stack_ok.final_estimator_.coef_.round(2)} → 평가 정확도 {acc_ok:.4f}")

기본값 트리 혼자   평가 정확도 : 0.6545
XGBoost 혼자       평가 정확도 : 0.7544
(X) 이미 본 데이터로 메타 학습 : 계수 [[13.87  0.25]] → 평가 정확도 0.6520
(O) out-of-fold 로 메타 학습   : 계수 [[0.07 5.08]] → 평가 정확도 0.7544


- (X) : 학습용에서 기본값 트리는 **거의 다 맞혀** 보이니까 메타 모델이 트리에 계수 **13.9** 를 준다 → 평가용에서는 트리가 0.65 짜리라 스태킹이 **0.652** 로 XGBoost 혼자(0.754)보다도 **나빠진다**.
- (O) : out-of-fold 에서는 트리의 진짜 실력이 드러나 계수가 **0.07** 로 줄고, XGBoost(5.08)를 믿는다 → 0.754.

## 3. 왜 스태킹이 별로 안 올랐나 — 기본 모델들이 **비슷한 말**을 한다

In [9]:
print("out-of-fold 확률 상관계수 :")
display(oof.corr().round(3))

# 세 모델의 0/1 예측이 엇갈리는 행은 얼마나?
p = pd.DataFrame({n: m.predict(X_test) for n, m in base_models})
disagree = (p.nunique(axis=1) > 1).mean()
print(f"평가용 14,139명 중 세 모델의 예측이 갈리는 비율 : {disagree:.1%}")

out-of-fold 확률 상관계수 :


,dt,rf,xgb
dt,1.000,0.957,0.952
rf,0.957,1.000,0.982
xgb,0.952,0.982,1.000


평가용 14,139명 중 세 모델의 예측이 갈리는 비율 : 9.9%


스태킹은 "서로 **다른** 실수를 하는 모델" 을 섞을 때 효과가 크다. 여기 세 모델은 모두 **트리 기반**이고 같은 데이터 · 같은 특성으로 배워서, 확률이 강하게 상관되고 예측이 갈리는 행이 적다 → 메타 모델이 고칠 여지가 거의 없다.
→ 교재 p.111 처럼 다양성을 늘리려면 **선형 모델 · KNN · SVM** 같은 다른 계열을 기본 모델에 섞는다.

## 4. 교재 p.111~113 버전 — 기본값 모델로 스태킹

In [10]:
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

base_estimators = [
    ('rf', RandomForestClassifier(random_state=0)),
    ('xgb', XGBClassifier(random_state=0)),
    ('dt', DecisionTreeClassifier(random_state=0)),
]
final_meta_learner = LogisticRegression(max_iter=1000, random_state=0)
stack_book = StackingClassifier(estimators=base_estimators, final_estimator=final_meta_learner, n_jobs=-1)
t0 = time.time()
stack_book.fit(X_train, y_train)
pred_test_book = stack_book.predict(X_test)        # 교재의 `stack.predict(X_test))` 는 괄호가 하나 더 있다
print(f"학습 {time.time()-t0:.0f}초 — 평가 정확도 {accuracy_score(y_test, pred_test_book):.4f}")
print("메타 계수 (rf, xgb, dt) :", stack_book.final_estimator_.coef_.round(3))

학습 25초 — 평가 정확도 0.7478
메타 계수 (rf, xgb, dt) : [[ 1.626  3.266 -0.01 ]]


## 5. 정리 — 모델 선택 기준 (교재 p.114)

In [11]:
final = pd.DataFrame([
    {'모델': '결정 트리 (튜닝)', '평가 정확도': accuracy_score(y_test, best_dt.predict(X_test)), '평가 F1': f1_score(y_test, best_dt.predict(X_test))},
    {'모델': '랜덤 포레스트 (튜닝)', '평가 정확도': accuracy_score(y_test, best_rf.predict(X_test)), '평가 F1': f1_score(y_test, best_rf.predict(X_test))},
    {'모델': 'XGBoost (튜닝)', '평가 정확도': accuracy_score(y_test, best_xgb.predict(X_test)), '평가 F1': f1_score(y_test, best_xgb.predict(X_test))},
    {'모델': '스태킹 (튜닝 3개 + 로지스틱)', '평가 정확도': accuracy_score(y_test, pred_test_stack), '평가 F1': f1_score(y_test, pred_test_stack)},
    {'모델': '스태킹 (교재, 기본값 3개)', '평가 정확도': accuracy_score(y_test, pred_test_book), '평가 F1': f1_score(y_test, pred_test_book)},
])
final.round(4)

,모델,평가 정확도,평가 F1
0,결정 트리 (튜닝),0.7411,0.7447
1,랜덤 포레스트 (튜닝),0.7519,0.7597
2,XGBoost (튜닝),0.7544,0.7629
3,스태킹 (튜닝 3개 + 로지스틱),0.7548,0.7605
4,"스태킹 (교재, 기본값 3개)",0.7478,0.7535


| 교재 p.114 | 이 데이터에서 |
|---|---|
| [1단계] 비즈니스 제약 — 설명 가능성이 필수인가? | 의료 데이터라 "왜 당뇨 위험인가" 설명이 중요 → 결정 트리 · 특성 중요도 |
| [2단계] 후보 모델 일괄 실험 | 트리 · 랜덤 포레스트 · XGBoost · 스태킹 |
| [3단계] 정량 지표 — 분류는 **F1** | 정확도는 스태킹 0.7548 ≈ XGBoost 0.7544, **F1 은 XGBoost(0.7629)가 더 높다**. 스태킹은 학습이 **6배** 비싸고 설명이 어렵다 → **XGBoost 하나**가 합리적 |

> **📝 고친 부분** (`분류 모델 성능 고도화 실습 1`)
> - 메타 모델 설명 "1단계 모델들의 예측값을 받아" → 정확히는 **out-of-fold 확률**(교차 검증으로 안 보고 낸 예측). 교재 p.109 "학습용 데이터에 대한 예측 결과" 를 그대로 쓰면 2-1 처럼 과하게 믿게 된다.
> - `base_models` 에 이미 학습된 `best_dt` 등을 넣었지만 `StackingClassifier` 는 **복제(clone)해서 처음부터 다시 학습**한다 → 기본 모델마다 5(교차 검증) + 1(재학습) = **6번** 학습. 이미 학습된 모델을 그대로 쓰는 게 아니다.
> - `from sklearn.metrics import accuracy_score, classification_report` 후 `accuracy_score` 는 안 씀 → 정확도도 함께 비교하도록 표로 정리.
> - 교재 p.112 `pred_test_stack = stack.predict(X_test))` → 닫는 괄호가 하나 더 있어 `SyntaxError`.